In [3]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from IPython.core.pylabtools import figsize
DB_PATH = "/Users/arnarfreyrerlingsson/Desktop/Polymarket_bitcoin_5min/db/polymarket.duckdb"

In [4]:
def query_to_df(query, limit=1000):
    if limit:
        limit = f"\n LIMIT {limit}"
    else:
        limit = f""
        #limit = f"\n LIMIT {HARD_LIMIT}"

    con = duckdb.connect(DB_PATH,read_only=True)
    try:
        query = str(query) + limit
        df = con.execute(query).df()
        return df
    finally:
        con.close()

In [5]:
query = "SHOW ALL TABLES"
query_to_df(query,None)

,database,schema,name,column_names,column_types,temporary
0,polymarket,main,MARKET_TRADES,"[THE_TIME, MARKET, PRICE, SIZE, SIDE, OUTCOME,...","[TIMESTAMP WITH TIME ZONE, VARCHAR, DOUBLE, DO...",False
1,polymarket,main,bitcoin_prices,"[OPEN_TIME, CLOSE_TIME, OPEN, HIGH, LOW, CLOSE...","[TIMESTAMP WITH TIME ZONE, TIMESTAMP WITH TIME...",False
2,polymarket,main,btc_history,"[SLUG, TOKEN_ID, OUTCOME, TIMESTAMP, PRICE]","[VARCHAR, VARCHAR, VARCHAR, TIMESTAMP WITH TIM...",False
3,polymarket,main,btc_summary,"[SLUG, TITLE, CONDITION_ID, UP_TOKEN_ID, DOWN_...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
4,polymarket,main,btc_trades,"[SLUG, CONDITION_ID, EVENT_SLUG, TITLE, ICON, ...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False


In [8]:
# Continuous ranges (no missing days) per table
tables = {
    "MARKET_TRADES": "THE_TIME",
    "bitcoin_prices": "OPEN_TIME",
    "btc_history": "TIMESTAMP",
    "btc_summary": "START",
    "btc_trades": "TIMESTAMP",
}
days = " UNION ALL ".join(
    f"SELECT '{t}' AS tbl, CAST({c} AS DATE) AS d, COUNT(*) AS n FROM {t} GROUP BY 2"
    for t, c in tables.items()
)
query = f"""
WITH days AS ({days}),
g AS (SELECT *, d - CAST(ROW_NUMBER() OVER (PARTITION BY tbl ORDER BY d) AS INT) AS grp FROM days)
SELECT tbl, MIN(d) AS first_day, MAX(d) AS last_day, COUNT(*) AS n_days, SUM(n) AS n_rows
FROM g GROUP BY tbl, grp ORDER BY tbl, first_day
"""
query_to_df(query, None)

,tbl,first_day,last_day,n_days,n_rows
0,MARKET_TRADES,2026-05-31,2026-09-29,122,73747459.0
1,bitcoin_prices,2026-06-01,2026-09-27,119,10281600.0
2,btc_history,2026-05-31,2026-09-29,122,4649292.0
3,btc_summary,2026-01-01,2026-01-26,26,7475.0
4,btc_summary,2026-06-01,2026-09-28,120,34560.0
5,btc_trades,2026-05-31,2026-09-29,122,73747459.0


### Market summary table

In [4]:
query = f"""

WITH MARKETS_SUMMARY AS (
SELECT
    SlUG AS MARKET,
    MARKET_ID,
    EVENT_STARTTIME AS EVENT_START,
    EVENT_END_DATE AS EVENT_CLOSE,
    PRICE_TO_BEAT,
    FINAL_PRICE,
    RESULT,VOLUME,
    N_TRADES,
    EVENT_START_DATE AS EVENT_OPEN

FROM btc_summary
WHERE PRICE_TO_BEAT IS NOT NULL
)
SELECT *
FROM MARKETS_SUMMARY
"""

#query_to_df(query)

### Market trades raw table

In [5]:
query = f"""
WITH MARKET_TRADES AS (
SELECT SlUG AS MARKET,TIMESTAMP AS THE_TIME,SECS_INTO_WINDOW,SIDE,OUTCOME,PRICE,SIZE,WALLET,NAME AS USER_NAME,
PSEUDONYM AS USER_PSUEDO
FROM btc_trades)

SELECT *
FROM MARKET_TRADES
ORDER BY THE_TIME ASC
"""
#query_to_df(query)


### Market trades main table

In [6]:
MARKET = 'btc-updown-5m-1780272000'

query = f"""
SELECT *
FROM MARKET_TRADES
"""

#query_to_df(query,1000)

,THE_TIME,MARKET,PRICE,SIZE,SIDE,OUTCOME,PRICE_TO_BEAT,FINAL_PRICE,RESULT,VOLUME,N_TRADES,SECS_INTO_WINDOW,EVENT_OPEN,EVENT_START,EVENT_CLOSE,WALLET,USER_NAME,USER_PSUEDO
0,2026-05-31 23:50:46+00:00,btc-updown-5m-1780272000,0.50,10.400000,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,-554,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0x0c4e9cc4f8987607d51fb761eb0c2696f442bf78,0x0C4e9cC4f8987607D51fb761Eb0c2696f442bF78-177...,Curvy-Laborer
1,2026-05-31 23:51:05+00:00,btc-updown-5m-1780272000,0.50,10.400000,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,-535,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0x0c4e9cc4f8987607d51fb761eb0c2696f442bf78,0x0C4e9cC4f8987607D51fb761Eb0c2696f442bF78-177...,Curvy-Laborer
2,2026-05-31 23:55:03+00:00,btc-updown-5m-1780272000,0.50,12.500000,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,-297,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0xced5a61d4e8c0e85df6d28351bba2d367664cd53,oyjhahaha,Damp-Recommendation
3,2026-05-31 23:55:05+00:00,btc-updown-5m-1780272000,0.50,5.000000,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,-295,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0x7ee1b149f81124ad566422b25bf23b8dab00a7f1,AdamOfEternia,Variable-Innovation
4,2026-05-31 23:55:07+00:00,btc-updown-5m-1780272000,0.51,9.800000,BUY,Up,73574.662155,73758.025335,Up,49500.575071,1900,-293,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0xa7ab95b9b75c905efbf6e1e65748dc4305c83734,mazala-,Humongous-Force
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2026-06-01 00:02:01+00:00,btc-updown-5m-1780272000,0.59,8.474575,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,121,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0x3295bd10cd5ceabcf626d1d484c7ed2200ed6f72,dpcruz,Superficial-Travel
996,2026-06-01 00:02:01+00:00,btc-updown-5m-1780272000,0.58,10.517240,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,121,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0xaffc3335c7a9679696adb348679264b87406a372,hello-cli,Spotted-Forest
997,2026-06-01 00:02:03+00:00,btc-updown-5m-1780272000,0.63,20.000000,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,123,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0x251c1a283703beed41590b0875a8dcb8ddd1541f,0xAAAAA,Suburban-Characterization
998,2026-06-01 00:02:03+00:00,btc-updown-5m-1780272000,0.64,4.330000,BUY,Down,73574.662155,73758.025335,Up,49500.575071,1900,123,2026-05-31 00:18:50.912679+00:00,2026-06-01 00:00:00+00:00,2026-06-01 00:05:00+00:00,0xfb0f17657c9c24293b918adb86362a4d8fc90b02,aoe2gamer,Nice-Prejudice
